In [1]:
import os
import sys

import numpy as np
import torch
import coremltools as ct

sys.path.append("..")

import utils
import torchinfo

import warnings
warnings.filterwarnings("ignore")


Torch version 2.5.0+cu118 has not been tested with coremltools. You may run into unexpected errors. Torch 2.4.0 is the most recent version that has been tested.
Failed to load _MLModelProxy: No module named 'coremltools.libcoremlpython'


In [2]:
sal_path = '/mnt/DataLake/AlienBackup/access/eht_gaze_prediction/models/unisal.pth'
traind_models_path = '/mnt/DataLake/AlienBackup/access/eht_gaze_prediction/trained_models_combined'
basename = 'conv_lstm_past_15_future_10_window_15_20250316_162231'
ckpnt = 'model_epoch_5_val_loss_49.0553.pth'


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load saliency model
unisal_model = utils.load_unisal(sal_path, device)
# Load time series model
time_model, timepoints, future_steps = utils.load_time_model(traind_models_path, basename, ckpnt, device)

# Create dummy input for sal model
image_sequence = torch.randn(timepoints, 288, 384, 3).cpu().numpy()
print(image_sequence.shape)
print(image_sequence[0].shape)
processed_imgs = []
for im_idx in range(timepoints):
    processed_imgs.append(utils.preprocess_sal_img(image_sequence[im_idx]))
processed_imgs = torch.stack(processed_imgs).to(device)
# print(processed_imgs[0].shape)
torchinfo.summary(unisal_model, processed_imgs[0].unsqueeze(0).shape)
# Convert Salincy model to CoreMl and save it to save folder


# Create dummy input for time series model
time_series_data = torch.randn(1, timepoints, 4).to(device)  # 4 is the number of original time features

sal_maps = []
with torch.no_grad():
    for img in processed_imgs:
        sal_maps.append(unisal_model(img.unsqueeze(0)).squeeze(0).squeeze(0).squeeze(0))
print(unisal_model(img.unsqueeze(0)).shape)

processed_maps = torch.stack(sal_maps).unsqueeze(0).to(device)
print(processed_maps.shape)
print(time_series_data.shape)


(15, 288, 384, 3)
(288, 384, 3)
torch.Size([1, 1, 1, 288, 384])
torch.Size([1, 15, 288, 384])
torch.Size([1, 15, 4])
